In [0]:
print("02_Bronze")

In [0]:
# ============================================================
# CITYCARE HEALTHCARE DATA PLATFORM
# 4.15J - BRONZE INGESTION
# ============================================================

from datetime import datetime
import uuid
import hashlib

from pyspark.sql import functions as F


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

REPO_ROOT = (
    "/Workspace/Repos/"
    "chandrakanthab@gmail.com/"
    "healthcare-data-platform"
)

EMR_SOURCE_PATH = (
    f"{REPO_ROOT}/sample-data/emr"
)

BRONZE_SCHEMA = "healthcare_bronze"

print("Bronze ingestion configuration loaded")
print(f"Source path : {EMR_SOURCE_PATH}")
print(f"Bronze schema: {BRONZE_SCHEMA}")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS healthcare_bronze;

In [0]:
%sql
SHOW SCHEMAS;

In [0]:
# ============================================================
# PIPELINE RUN METADATA
# ============================================================

pipeline_run_id = str(uuid.uuid4())

batch_id = (
    "BATCH-"
    + datetime.now().strftime("%Y%m%d%H%M%S")
)

ingestion_timestamp = datetime.now()

print(f"Pipeline Run ID : {pipeline_run_id}")
print(f"Batch ID        : {batch_id}")
print(f"Ingestion Time  : {ingestion_timestamp}")

In [0]:
def ingest_csv_to_bronze(
    source_path,
    table_name,
    source_system,
    pipeline_run_id,
    batch_id,
    ingestion_timestamp
):
    """
    Read a source CSV and write it as a Bronze Delta table.

    Bronze responsibilities:
    - Preserve source columns
    - Add technical metadata
    - Write Delta
    """

    print()
    print("=" * 60)
    print(f"INGESTING: {table_name}")
    print("=" * 60)

    # --------------------------------------------------------
    # Read source
    # --------------------------------------------------------

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(source_path)
    )

    source_file_name = source_path.split("/")[-1]

    # --------------------------------------------------------
    # Add audit metadata
    # --------------------------------------------------------

    df = (
        df
        .withColumn(
            "source_system",
            F.lit(source_system)
        )
        .withColumn(
            "source_file_name",
            F.lit(source_file_name)
        )
        .withColumn(
            "ingestion_timestamp",
            F.lit(ingestion_timestamp)
        )
        .withColumn(
            "pipeline_run_id",
            F.lit(pipeline_run_id)
        )
        .withColumn(
            "batch_id",
            F.lit(batch_id)
        )
        .withColumn(
            "ingestion_date",
            F.to_date(
                F.lit(ingestion_timestamp)
            )
        )
    )

    # --------------------------------------------------------
    # Add record hash
    # --------------------------------------------------------

    business_columns = [
        column
        for column in df.columns
        if column not in [
            "source_system",
            "source_file_name",
            "ingestion_timestamp",
            "pipeline_run_id",
            "batch_id",
            "ingestion_date",
            "record_hash"
        ]
    ]

    df = df.withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(
                        F.col(column).cast("string"),
                        F.lit("")
                    )
                    for column in business_columns
                ]
            ),
            256
        )
    )

    # --------------------------------------------------------
    # Write Delta
    # --------------------------------------------------------

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            f"{BRONZE_SCHEMA}.{table_name}"
        )
    )

    # --------------------------------------------------------
    # Audit output
    # --------------------------------------------------------

    record_count = df.count()

    print(
        f"✓ Source records : {record_count}"
    )

    print(
        f"✓ Bronze table   : "
        f"{BRONZE_SCHEMA}.{table_name}"
    )

    print(
        "✓ Bronze ingestion successful"
    )

    return record_count

In [0]:
# ============================================================
# ALL HEALTHCARE SOURCE TABLES
# ============================================================

source_tables = [

    # EMR
    {
        "source_system": "EMR",
        "source_path": f"{REPO_ROOT}/sample-data/emr/patient.csv",
        "bronze_table": "emr_patient"
    },
    {
        "source_system": "EMR",
        "source_path": f"{REPO_ROOT}/sample-data/emr/provider.csv",
        "bronze_table": "emr_provider"
    },
    {
        "source_system": "EMR",
        "source_path": f"{REPO_ROOT}/sample-data/emr/department.csv",
        "bronze_table": "emr_department"
    },
    {
        "source_system": "EMR",
        "source_path": f"{REPO_ROOT}/sample-data/emr/appointment.csv",
        "bronze_table": "emr_appointment"
    },
    {
        "source_system": "EMR",
        "source_path": f"{REPO_ROOT}/sample-data/emr/encounter.csv",
        "bronze_table": "emr_encounter"
    },

    # LIS
    {
        "source_system": "LIS",
        "source_path": f"{REPO_ROOT}/sample-data/lis/test_master.csv",
        "bronze_table": "lis_test_master"
    },
    {
        "source_system": "LIS",
        "source_path": f"{REPO_ROOT}/sample-data/lis/lab_order.csv",
        "bronze_table": "lis_lab_order"
    },
    {
        "source_system": "LIS",
        "source_path": f"{REPO_ROOT}/sample-data/lis/lab_result.csv",
        "bronze_table": "lis_lab_result"
    },

    # Pharmacy
    {
        "source_system": "PHARMACY",
        "source_path": f"{REPO_ROOT}/sample-data/pharmacy/medication_master.csv",
        "bronze_table": "pharmacy_medication_master"
    },
    {
        "source_system": "PHARMACY",
        "source_path": f"{REPO_ROOT}/sample-data/pharmacy/prescription.csv",
        "bronze_table": "pharmacy_prescription"
    },
    {
        "source_system": "PHARMACY",
        "source_path": f"{REPO_ROOT}/sample-data/pharmacy/dispensing.csv",
        "bronze_table": "pharmacy_dispensing"
    },

    # Billing
    {
        "source_system": "BILLING",
        "source_path": f"{REPO_ROOT}/sample-data/billing/invoice.csv",
        "bronze_table": "billing_invoice"
    },
    {
        "source_system": "BILLING",
        "source_path": f"{REPO_ROOT}/sample-data/billing/invoice_line.csv",
        "bronze_table": "billing_invoice_line"
    },
    {
        "source_system": "BILLING",
        "source_path": f"{REPO_ROOT}/sample-data/billing/payment.csv",
        "bronze_table": "billing_payment"
    }
]


# ============================================================
# INGEST ALL SOURCE SYSTEMS
# ============================================================

ingestion_summary = []

for source in source_tables:

    count = ingest_csv_to_bronze(
        source_path=source["source_path"],
        table_name=source["bronze_table"],
        source_system=source["source_system"],
        pipeline_run_id=pipeline_run_id,
        batch_id=batch_id,
        ingestion_timestamp=ingestion_timestamp
    )

    ingestion_summary.append(
        {
            "source_system": source["source_system"],
            "bronze_table": source["bronze_table"],
            "record_count": count
        }
    )


ingestion_summary_df = spark.createDataFrame(
    ingestion_summary
)

display(
    ingestion_summary_df
)

In [0]:
%sql
SELECT COUNT(*) AS patient_count
FROM healthcare_bronze.emr_patient;

SELECT
    source_system,
    source_file_name,
    pipeline_run_id,
    batch_id,
    ingestion_timestamp,
    COUNT(*) AS records
FROM healthcare_bronze.emr_patient
GROUP BY
    source_system,
    source_file_name,
    pipeline_run_id,
    batch_id,
    ingestion_timestamp;

In [0]:
patient_source = spark.read.option(
    "header", True
).option(
    "inferSchema", True
).csv(
    f"{REPO_ROOT}/sample-data/emr/patient.csv"
)

print("Current patient.csv records:", patient_source.count())

In [0]:
%sql
SELECT
    'emr_patient' AS table_name,
    COUNT(*) AS record_count
FROM healthcare_bronze.emr_patient

UNION ALL

SELECT
    'emr_provider',
    COUNT(*)
FROM healthcare_bronze.emr_provider

UNION ALL

SELECT
    'emr_department',
    COUNT(*)
FROM healthcare_bronze.emr_department

UNION ALL

SELECT
    'emr_appointment',
    COUNT(*)
FROM healthcare_bronze.emr_appointment

UNION ALL

SELECT
    'emr_encounter',
    COUNT(*)
FROM healthcare_bronze.emr_encounter;

In [0]:
bronze_tables = [
    ("EMR", "patient", "emr_patient"),
    ("EMR", "provider", "emr_provider"),
    ("EMR", "department", "emr_department"),
    ("EMR", "appointment", "emr_appointment"),
    ("EMR", "encounter", "emr_encounter"),

    ("LIS", "test_master", "lis_test_master"),
    ("LIS", "lab_order", "lis_lab_order"),
    ("LIS", "lab_result", "lis_lab_result"),

    ("PHARMACY", "medication_master", "pharmacy_medication_master"),
    ("PHARMACY", "prescription", "pharmacy_prescription"),
    ("PHARMACY", "dispensing", "pharmacy_dispensing"),

    ("BILLING", "invoice", "billing_invoice"),
    ("BILLING", "invoice_line", "billing_invoice_line"),
    ("BILLING", "payment", "billing_payment")
]

reconciliation_results = []

for source_system, source_table, bronze_table in bronze_tables:

    source_path = (
        f"{REPO_ROOT}/sample-data/"
        f"{source_system.lower()}/{source_table}.csv"
    )

    source_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(source_path)
    )

    bronze_df = spark.table(
        f"{BRONZE_SCHEMA}.{bronze_table}"
    )

    source_count = source_df.count()
    bronze_count = bronze_df.count()

    reconciliation_results.append({
        "source_system": source_system,
        "source_table": source_table,
        "bronze_table": bronze_table,
        "source_count": source_count,
        "bronze_count": bronze_count,
        "count_difference": source_count - bronze_count,
        "status": (
            "PASS"
            if source_count == bronze_count
            else "FAIL"
        )
    })

reconciliation_df = spark.createDataFrame(
    reconciliation_results
)

display(reconciliation_df)